In [1]:
# Core imports for the whole lab
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

np.random.seed(42)
print('Setup complete. NumPy', np.__version__)

Setup complete. NumPy 2.1.3


# 1. Probability theory & **axioms**

In [2]:
# -----------------------------------------------------------
# 🔹 1A. PROBABILITY BY SIMULATION
# -----------------------------------------------------------

# P(A) = favourable outcomes / total outcomes.
# We can estimate it by simulating many trials.
rolls = np.random.randint(1, 7, size=100_000)   # 100k dice rolls

p_even = (rolls % 2 == 0).mean()       # P(even)
p_gt4  = (rolls > 4).mean()            # P(roll > 4)
print('P(even) ~', round(p_even, 3), ' (true 0.5)')
print('P(>4)   ~', round(p_gt4, 3),  ' (true 0.333)')


P(even) ~ 0.502  (true 0.5)
P(>4)   ~ 0.334  (true 0.333)


In [3]:
# -----------------------------------------------------------
# 🔹 1B. THE ADDITION RULE (disjoint events)
# -----------------------------------------------------------

# Rolling a 1 OR a 2 — these events can't both happen (disjoint)
p_1 = (rolls == 1).mean()
p_2 = (rolls == 2).mean()
p_1_or_2 = (np.isin(rolls, [1, 2])).mean()
print('P(1) + P(2) =', round(p_1 + p_2, 3))
print('P(1 or 2)   =', round(p_1_or_2, 3), ' -> they match')


P(1) + P(2) = 0.334
P(1 or 2)   = 0.334  -> they match


In [7]:
"""
NASSCOM AI Workshop: uc-0a (Complaint Classifier)
Implements the RICE & CRAFT software frameworks for vibe-coded evaluation.
"""
import os
import json

def classify_complaint(text: str) -> dict:
    """
    Evaluates raw civic complaints and routes them to correct city departments.
    """
    # Clean input
    clean_text = text.strip().lower()

    # Baseline deterministic heuristic (Replace/augment with LLM API orchestration as needed)
    if any(word in clean_text for word in ["pothole", "road", "street", "tarmac"]):
        department = "Public Works Department"
        priority = "High"
    elif any(word in clean_text for word in ["water", "sewage", "leak", "drain"]):
        department = "Water Supply & Sewerage Board"
        priority = "Critical"
    elif any(word in clean_text for word in ["garbage", "trash", "waste", "dump"]):
        department = "Solid Waste Management"
        priority = "Medium"
    else:
        department = "General Administration"
        priority = "Low"

    return {
        "text_analyzed": text,
        "assigned_department": department,
        "priority_level": priority,
        "status": "Ready for Routing"
    }

if __name__ == "__main__":
    # Test sample simulating nasscom city-test-files data
    sample_complaint = "There is a massive water leak leaking all over the main road near the market."
    result = classify_complaint(sample_complaint)
    print(json.dumps(result, indent=4))


{
    "text_analyzed": "There is a massive water leak leaking all over the main road near the market.",
    "assigned_department": "Public Works Department",
    "priority_level": "High",
    "status": "Ready for Routing"
}


# **2. Conditional probability & independence**

In [ ]:
# -----------------------------------------------------------
# 🔹 2A. CONDITIONAL PROBABILITY  P(A | B) = P(A and B) / P(B)
# -----------------------------------------------------------

rolls = np.random.randint(1, 7, size=100_000)

# P(roll is 6 | roll is even):  narrow the world to even rolls
even = rolls[rolls % 2 == 0]          # condition on 'even'
p_6_given_even = (even == 6).mean()
print('P(6 | even) ~', round(p_6_given_even, 3), ' (true 1/3)')


In [ ]:
# -----------------------------------------------------------
# 🔹 2B. TESTING FOR INDEPENDENCE
# -----------------------------------------------------------

# Two events are independent if P(A|B) == P(A).
# 'roll > 3' and 'roll is even' -> are they independent?
A = rolls > 3
B = rolls % 2 == 0
p_A        = A.mean()
p_A_given_B = A[B].mean()
print('P(A)      =', round(p_A, 3))
print('P(A | B)  =', round(p_A_given_B, 3))
print('Independent?', np.isclose(p_A, p_A_given_B, atol=0.02))


In [9]:
# 1. P(odd | roll < 4)  -> condition on rolls < 4, then check odd
less_than_4 = rolls[rolls < 4]
p_odd_given_less_than_4 = (less_than_4 % 2 != 0).mean()

# 2. P(roll < 4)
p_less_than_4 = (rolls < 4).mean()

# 3. Compare P(odd | <4) with P(odd) overall — independent?
p_odd_overall = (rolls % 2 != 0).mean()

# Check independence (Mathematically, they are independent if P(A|B) == P(A))
# Because of simulation variance, we check if they are approximately equal
is_independent = np.isclose(p_odd_given_less_than_4, p_odd_overall, atol=0.01)


# 3. Bayes' **theorem**

In [ ]:
 -----------------------------------------------------------
# 🔹 3A. THE MEDICAL-TEST PROBLEM (by formula)
# -----------------------------------------------------------

# Disease affects 1% of people; test is 99% accurate.
p_disease   = 0.01                       # prior  P(D)
p_pos_given_D  = 0.99                     # true positive rate  P(+|D)
p_pos_given_nD = 0.01                     # false positive rate P(+|not D)

# P(+) = P(+|D)P(D) + P(+|notD)P(notD)   (total probability)
p_pos = p_pos_given_D * p_disease + p_pos_given_nD * (1 - p_disease)

# Bayes: P(D | +)
p_D_given_pos = (p_pos_given_D * p_disease) / p_pos
print('P(sick | positive test) =', round(p_D_given_pos, 3))
print('-> only ~50%, because the disease is rare (base-rate effect)')

In [ ]:
# -----------------------------------------------------------
# 🔹 3B. THE SAME ANSWER BY SIMULATION (sanity check)
# -----------------------------------------------------------

N = 1_000_000
has_disease = np.random.rand(N) < p_disease
# test result depends on disease status
tests_pos = np.where(has_disease,
                     np.random.rand(N) < p_pos_given_D,    # sick -> 99% positive
                     np.random.rand(N) < p_pos_given_nD)   # healthy -> 1% positive

among_positive = has_disease[tests_pos]      # of those who tested positive...
print('Simulated P(sick | positive) =', round(among_positive.mean(), 3))


In [ ]:
# 1. priors / likelihoods
p_spam = 0.20
p_free_given_spam = 0.60
p_free_given_ham  = 0.05

# 2. P('free') via total probability
# P('free') = P('free' | spam) * P(spam) + P('free' | ham) * P(ham)
p_ham = 1.0 - p_spam
p_free = (p_free_given_spam * p_spam) + (p_free_given_ham * p_ham)

# 3. Bayes: P(spam | 'free')
# P(spam | 'free') = (P('free' | spam) * P(spam)) / P('free')
p_spam_given_free = (p_free_given_spam * p_spam) / p_free


# 4. Probability **distributions**

In [ ]:
# -----------------------------------------------------------
# 🔹 4A. BERNOULLI & POISSON (discrete)
# -----------------------------------------------------------

# Bernoulli: a single yes/no trial with probability p
bern = stats.bernoulli(p=0.3).rvs(size=10_000)
print('Bernoulli(0.3) mean ~', round(bern.mean(), 3), '(true 0.3)')

# Poisson: count of events per interval, rate lambda
pois = stats.poisson(mu=4).rvs(size=10_000)
print('Poisson(4) mean ~', round(pois.mean(), 3), '(true 4)')



In [ ]:
# -----------------------------------------------------------
# 🔹 4B. THE GAUSSIAN (normal) — plot the bell curve
# -----------------------------------------------------------

x = np.linspace(-5, 5, 200)
fig, ax = plt.subplots(figsize=(7, 3.5))
for sigma in [0.5, 1.0, 2.0]:
    ax.plot(x, stats.norm(0, sigma).pdf(x), label=f'sigma={sigma}')
ax.set_title('Gaussian PDF for different spreads'); ax.legend(); plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import scipy.stats as stats

# 1. Poisson(mu=2) samples + mean
# In numpy, 'lam' represents the rate parameter mu (expected value)
poisson_samples = np.random.poisson(lam=2, size=10000)
print("Sample Mean:", poisson_samples.mean())

# 2. histogram of the samples (plt.hist(...))
# Since Poisson data is discrete, we adjust bins to align with integers
plt.figure(figsize=(8, 4))
plt.hist(poisson_samples, bins=np.arange(0, poisson_samples.max() + 2) - 0.5, rwidth=0.8, density=True, alpha=0.75, edgecolor='black')
plt.title("Histogram of Poisson Samples (mu=2)")
plt.xlabel("Value")
plt.ylabel("Density")
plt.grid(axis='y', alpha=0.5)
plt.show()

# 3. Gaussian PDF for two sigmas on one plot
x = np.linspace(-10, 10, 500)
mu = 0
sigma1 = 1.0
sigma2 = 3.0

pdf1 = stats.norm.pdf(x, mu, sigma1)
pdf2 = stats.norm.pdf(x, mu, sigma2)

plt.figure(figsize=(8, 4))
plt.plot(x, pdf1, label=f'mu={mu}, sigma={sigma1} (Narrow)', color='royalblue', linewidth=2)
plt.plot(x, pdf2, label=f'mu={mu}, sigma={sigma2} (Wide)', color='orange', linewidth=2)
plt.title("Gaussian PDFs with Different Standard Deviations")
plt.xlabel("x")
plt.ylabel("Probability Density")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()


# 5. Entropy, KL divergence & mutual **information**

In [ ]:

# -----------------------------------------------------------
# 🔹 5A. ENTROPY — how uncertain is a distribution?
# -----------------------------------------------------------

from scipy.stats import entropy

fair   = [0.5, 0.5]      # a fair coin: maximum uncertainty
biased = [0.9, 0.1]      # a biased coin: more predictable
certain= [1.0, 0.0]      # no uncertainty at all

# base=2 gives entropy in BITS
print('Entropy fair coin   :', round(entropy(fair, base=2), 3), 'bits')
print('Entropy biased coin :', round(entropy(biased, base=2), 3), 'bits')
print('Entropy certain     :', round(entropy(certain, base=2), 3), 'bits')


In [ ]:
# -----------------------------------------------------------
# 🔹 5B. KL DIVERGENCE — how far is Q from P?
# -----------------------------------------------------------

P = np.array([0.5, 0.5])
Q = np.array([0.9, 0.1])

# scipy's entropy(P, Q) computes the KL divergence D(P || Q)
print('D(P || Q) =', round(entropy(P, Q, base=2), 3), 'bits')
print('D(P || P) =', round(entropy(P, P, base=2), 3), '-> zero (identical)')
print('Note: D(P||Q) != D(Q||P)  ->', round(entropy(Q, P, base=2), 3), '(not symmetric)')

In [ ]:
# -----------------------------------------------------------
# 🔹 5C. MUTUAL INFORMATION — which feature is most informative?
# -----------------------------------------------------------

from sklearn.feature_selection import mutual_info_classif
from sklearn.datasets import load_iris

iris = load_iris()
mi = mutual_info_classif(iris.data, iris.target, random_state=0)
for name, score in sorted(zip(iris.feature_names, mi), key=lambda t: -t[1]):
    print(f'{score:.3f}  {name}')
print('-> higher MI = the feature tells us more about the class')


In [ ]:
import numpy as np

# 1. Entropy of a fair 4-sided and 6-sided die (use base=2)
# H(X) = log2(n) for a uniform distribution
entropy_4 = np.log2(4)
entropy_6 = np.log2(6)

more_uncertain = "6-sided die" if entropy_6 > entropy_4 else "4-sided die"

# 2. KL divergence D(P || Q)
P = np.array([0.7, 0.3])
Q = np.array([0.5, 0.5])
kl_divergence = np.sum(P * np.log2(P / Q))

# 3. Most / least informative iris feature (from standard Iris MI scores)
# Note: Based on standard Scikit-Learn mutual_info_classif benchmarks for the Iris dataset:
most_informative = "petal length (cm)"  # Or petal width (cm), both score ~0.99
least_informative = "sepal width (cm)"   # Scores lowest, typically ~0.22 - 0.30
